# 1. Setup & Imports
This section imports required libraries, configures paths, and defines the experiment scope for FinSight XGBoost forecasting.

In [1]:
import sys
sys.path.append("../../../")

import json
from pathlib import Path

import joblib
import numpy as np
import optuna
import pandas as pd
import plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor

from backend.data.data_loader import fetch_stock_data
from backend.data.preprocessor import FinSightPreprocessor

optuna.logging.set_verbosity(optuna.logging.INFO)

TICKER = "SBIN.NS"
START = "2020-01-01"
END = "2026-05-01"
TEST_START = "2025-01-01"

safe_ticker = ''.join(ch if ch.isalnum() else '_' for ch in TICKER)
ARTIFACTS_DIR = Path("./")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Ticker: {TICKER}")
print(f"Date range: {START} to {END}")
print(f"Artifacts dir: {ARTIFACTS_DIR.resolve()}")

c:\Users\gaura\Desktop\FinSight\finsightvenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ticker: SBIN.NS
Date range: 2020-01-01 to 2026-05-01
Artifacts dir: C:\Users\gaura\Desktop\FinSight\notebooks\models\xgboost


# 2. Load Data
Load OHLCV data using FinSight's shared data loader and inspect shape and schema.

In [2]:
df = fetch_stock_data(TICKER, START, END)
print("Shape:", df.shape)
display(df.head())
df.info()

Shape: (1567, 9)


Price,Open,High,Low,Close,Volume,Adj Close,daily_return,log_return,hl_spread
Date,,,,,,,,,
2020-01-02,334.500000,339.850006,333.350006,339.299988,20324236,307.534393,0.014501,0.014397,6.500000
2020-01-03,337.950012,337.950012,332.000000,333.700012,21853208,302.458710,-0.016504,-0.016642,5.950012
2020-01-06,331.700012,331.700012,317.700012,319.000000,35645325,289.134949,-0.044052,-0.045051,14.000000
2020-01-07,324.450012,327.000000,315.399994,318.399994,50966826,288.591095,-0.001881,-0.001883,11.600006
2020-01-08,312.100006,321.500000,311.000000,319.799988,44527485,289.859985,0.004397,0.004387,10.500000


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1567 entries, 2020-01-02 to 2026-05-01
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Open          1567 non-null   float64
 1   High          1567 non-null   float64
 2   Low           1567 non-null   float64
 3   Close         1567 non-null   float64
 4   Volume        1567 non-null   int64  
 5   Adj Close     1567 non-null   float64
 6   daily_return  1567 non-null   float64
 7   log_return    1567 non-null   float64
 8   hl_spread     1567 non-null   float64
dtypes: float64(8), int64(1)
memory usage: 122.4 KB


# 3. Feature Engineering
Apply FinSight preprocessing and optionally enrich the feature frame using graph features if the JSON artifact exists.
The target variable is next-day Close (`Close.shift(-1)`).

In [3]:
preprocessor = FinSightPreprocessor(ticker=TICKER)
processed_df = preprocessor.fit_transform(df)

graph_features_path = ARTIFACTS_DIR / "graph_features.json"
if graph_features_path.exists():
    with graph_features_path.open("r", encoding="utf-8") as f:
        graph_features = json.load(f)
    for k, v in graph_features.items():
        processed_df[k] = float(v)
    print(f"Loaded graph features from {graph_features_path}")
else:
    graph_features = {}
    print(f"No graph features found at {graph_features_path}; continuing without them.")

model_df = processed_df.copy()
model_df["target"] = model_df["Close"].shift(-1)
model_df = model_df.dropna(subset=["target"]).replace([np.inf, -np.inf], np.nan).dropna()

print("Final model frame shape:", model_df.shape)
display(model_df.head())
print("Feature columns (excluding target):", len([c for c in model_df.columns if c != "target"]))

No graph features found at graph_features.json; continuing without them.
Final model frame shape: (1547, 15)


,Open,High,Low,Close,Volume,Adj Close,daily_return,log_return,hl_spread,Close_lag_1,Close_lag_5,Close_lag_10,rolling_mean_20,rolling_std_20,target
Date,,,,,,,,,,,,,,,
2020-01-29,-1.123457,-1.133921,-1.112527,-1.127601,-0.060100,-1.119131,0.168870,0.177542,-1.022843,-1.132165,-1.124108,-1.085850,-1.095752,-0.861506,-1.151450
2020-01-30,-1.128020,-1.146083,-1.153875,-1.151450,0.431931,-1.140636,-0.979159,-0.973986,-0.202971,-1.126561,-1.094723,-1.089832,-1.101759,-0.920186,-1.119306
2020-01-31,-1.140671,-1.125676,-1.141554,-1.119306,2.785020,-1.111651,1.225108,1.214481,0.032964,-1.150431,-1.091180,-1.112048,-1.104963,-0.979974,-1.203710
2020-02-03,-1.185054,-1.186897,-1.196893,-1.203710,1.270363,-1.187759,-3.317324,-3.403017,-0.155785,-1.118259,-1.123900,-1.128814,-1.109353,-0.809970,-1.169907
2020-02-04,-1.185469,-1.184012,-1.189584,-1.169907,1.066792,-1.157279,1.347613,1.333393,-0.279653,-1.202736,-1.128485,-1.130071,-1.111905,-0.759029,-1.150206


Feature columns (excluding target): 14


# 4. Train/Test Split
Create a chronological split with the last 20% as test data (no shuffling).

In [4]:
train_df = model_df[model_df.index < "2025-01-01"]
test_df = model_df[model_df.index >= "2025-01-01"]

feature_cols = [c for c in model_df.columns if c != "target"]
X_train, y_train = train_df[feature_cols], train_df["target"]
X_test, y_test = test_df[feature_cols], test_df["target"]

print("Train size:", len(train_df))
print("Test size:", len(test_df))
print("Features:", len(feature_cols))

Train size: 1218
Test size: 329
Features: 14


# 5. Time Series Cross Validation
Run baseline cross-validation using `TimeSeriesSplit(n_splits=5)` and report fold RMSE.

In [5]:
tscv = TimeSeriesSplit(n_splits=5)
baseline_rmses = []

for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train), start=1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    model = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        objective="reg:squarederror",
        tree_method="hist",
        device="cuda"
    )
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=100)

    preds = model.predict(X_val)
    rmse = float(np.sqrt(mean_squared_error(y_val, preds)))
    baseline_rmses.append(rmse)
    print(f"Fold {fold} RMSE: {rmse:.6f}")

print(f"Mean CV RMSE: {np.mean(baseline_rmses):.6f}")

[0]	validation_0-rmse:0.70449
[100]	validation_0-rmse:0.36748
[200]	validation_0-rmse:0.36465
[300]	validation_0-rmse:0.36434
[400]	validation_0-rmse:0.36436
[499]	validation_0-rmse:0.36436
Fold 1 RMSE: 0.364357
[0]	validation_0-rmse:0.79016


c:\Users\gaura\Desktop\FinSight\finsightvenv\lib\site-packages\xgboost\core.py:751: UserWarning: [00:55:50] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[100]	validation_0-rmse:0.22199
[200]	validation_0-rmse:0.21527
[300]	validation_0-rmse:0.21503
[400]	validation_0-rmse:0.21497
[499]	validation_0-rmse:0.21486
Fold 2 RMSE: 0.214855
[0]	validation_0-rmse:0.82015
[100]	validation_0-rmse:0.22305
[200]	validation_0-rmse:0.21715
[300]	validation_0-rmse:0.21709
[400]	validation_0-rmse:0.21727
[499]	validation_0-rmse:0.21733
Fold 3 RMSE: 0.217334
[0]	validation_0-rmse:0.82698
[100]	validation_0-rmse:0.19362
[200]	validation_0-rmse:0.18974
[300]	validation_0-rmse:0.19003
[400]	validation_0-rmse:0.18996
[499]	validation_0-rmse:0.18996
Fold 4 RMSE: 0.189962
[0]	validation_0-rmse:1.47554
[100]	validation_0-rmse:0.36786
[200]	validation_0-rmse:0.35308
[300]	validation_0-rmse:0.35087
[400]	validation_0-rmse:0.34910
[499]	validation_0-rmse:0.34846
Fold 5 RMSE: 0.348456
Mean CV RMSE: 0.266993


# 6. Hyperparameter Tuning (Optuna)
Tune XGBoost hyperparameters with 50 Optuna trials using time-series CV and GPU execution.

In [6]:
def objective(trial: optuna.Trial) -> float:
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "random_state": 42,
        "objective": "reg:squarederror",
        "tree_method": "hist",
        "device": "cuda"
    }

    cv = TimeSeriesSplit(n_splits=3)
    rmses = []
    for tr_idx, val_idx in cv.split(X_train):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        model = XGBRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=100)
        pred = model.predict(X_val)
        rmse = float(np.sqrt(mean_squared_error(y_val, pred)))
        rmses.append(rmse)

    print(f"Trial {trial.number} | RMSE: {float(np.mean(rmses)):.4f} | params: {trial.params}")
    return float(np.mean(rmses))

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

best_params = study.best_params
print("Best RMSE:", study.best_value)
print("Best params:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

[I 2026-05-24 00:55:55,910] A new study created in memory with name: no-name-06bb6f0b-85e7-4729-9acc-1eeeda23c503


[0]	validation_0-rmse:0.82017
[100]	validation_0-rmse:0.32840
[200]	validation_0-rmse:0.32950
[300]	validation_0-rmse:0.32968
[400]	validation_0-rmse:0.32982
[427]	validation_0-rmse:0.32984
[0]	validation_0-rmse:0.78817
[100]	validation_0-rmse:0.20993
[200]	validation_0-rmse:0.21049
[300]	validation_0-rmse:0.21041
[400]	validation_0-rmse:0.21049
[427]	validation_0-rmse:0.21059
[0]	validation_0-rmse:1.30863
[100]	validation_0-rmse:0.70109
[200]	validation_0-rmse:0.69941
[300]	validation_0-rmse:0.70015
[400]	validation_0-rmse:0.70027
[427]	validation_0-rmse:0.70033


[I 2026-05-24 00:55:58,611] Trial 0 finished with value: 0.41358760765020336 and parameters: {'n_estimators': 428, 'learning_rate': 0.1607801893044173, 'max_depth': 5, 'subsample': 0.8420352358992049, 'colsample_bytree': 0.9556117733863135, 'min_child_weight': 6}. Best is trial 0 with value: 0.41358760765020336.


Trial 0 | RMSE: 0.4136 | params: {'n_estimators': 428, 'learning_rate': 0.1607801893044173, 'max_depth': 5, 'subsample': 0.8420352358992049, 'colsample_bytree': 0.9556117733863135, 'min_child_weight': 6}
[0]	validation_0-rmse:0.77022
[100]	validation_0-rmse:0.32962
[200]	validation_0-rmse:0.32918
[232]	validation_0-rmse:0.32904
[0]	validation_0-rmse:0.71869
[100]	validation_0-rmse:0.21062
[200]	validation_0-rmse:0.21231
[232]	validation_0-rmse:0.21285
[0]	validation_0-rmse:1.23186
[100]	validation_0-rmse:0.70765
[200]	validation_0-rmse:0.70844
[232]	validation_0-rmse:0.70816


[I 2026-05-24 00:56:00,676] Trial 1 finished with value: 0.41668603648473207 and parameters: {'n_estimators': 233, 'learning_rate': 0.27513363195933305, 'max_depth': 7, 'subsample': 0.6622648378165674, 'colsample_bytree': 0.8299493754795515, 'min_child_weight': 10}. Best is trial 0 with value: 0.41358760765020336.


...

Trial 17 | RMSE: 0.4157 | params: {'n_estimators': 779, 'learning_rate': 0.0669640730320186, 'max_depth': 3, 'subsample': 0.6634631055607404, 'colsample_bytree': 0.690950326706307, 'min_child_weight': 9}
[0]	validation_0-rmse:0.84903
[100]	validation_0-rmse:0.32928
[200]	validation_0-rmse:0.32690
[300]	validation_0-rmse:0.32563
[400]	validation_0-rmse:0.32638
[500]	validation_0-rmse:0.32612
[600]	validation_0-rmse:0.32550
[700]	validation_0-rmse:0.32583
[800]	validation_0-rmse:0.32581
[900]	validation_0-rmse:0.32567
[955]	validation_0-rmse:0.32571
[0]	validation_0-rmse:0.82089
[100]	validation_0-rmse:0.21549
[200]	validation_0-rmse:0.21321
[300]	validation_0-rmse:0.21374
[400]	validation_0-rmse:0.21455
[500]	validation_0-rmse:0.21430
[600]	validation_0-rmse:0.21446
[700]	validation_0-rmse:0.21467
[800]	validation_0-rmse:0.21497
[900]	validation_0-rmse:0.21511
[955]	validation_0-rmse:0.21475
[0]	validation_0-rmse:1.34539
[100]	validation_0-rmse:0.71351
[200]	validation_0-rmse:0.71273
[3

[I 2026-05-24 00:59:42,650] Trial 18 finished with value: 0.4169749448519404 and parameters: {'n_estimators': 956, 'learning_rate': 0.10688636120483759, 'max_depth': 4, 'subsample': 0.7464523033345919, 'colsample_bytree': 0.7658261876125685, 'min_child_weight': 10}. Best is trial 16 with value: 0.41048199150367504.


Trial 18 | RMSE: 0.4170 | params: {'n_estimators': 956, 'learning_rate': 0.10688636120483759, 'max_depth': 4, 'subsample': 0.7464523033345919, 'colsample_bytree': 0.7658261876125685, 'min_child_weight': 10}
[0]	validation_0-rmse:0.81042
[100]	validation_0-rmse:0.32861
[200]	validation_0-rmse:0.32988
[300]	validation_0-rmse:0.32999
[400]	validation_0-rmse:0.32995
[500]	validation_0-rmse:0.33007
[600]	validation_0-rmse:0.33005
[604]	validation_0-rmse:0.33005
[0]	validation_0-rmse:0.76846
[100]	validation_0-rmse:0.22174
[200]	validation_0-rmse:0.22706
[300]	validation_0-rmse:0.22816
[400]	validation_0-rmse:0.22865
[500]	validation_0-rmse:0.22868
[600]	validation_0-rmse:0.22869
[604]	validation_0-rmse:0.22864
[0]	validation_0-rmse:1.28695
[100]	validation_0-rmse:0.70113
[200]	validation_0-rmse:0.70284
[300]	validation_0-rmse:0.70331
[400]	validation_0-rmse:0.70340
[500]	validation_0-rmse:0.70341
[600]	validation_0-rmse:0.70343
[604]	validation_0-rmse:0.70343


[I 2026-05-24 01:00:57,318] Trial 19 finished with value: 0.42070671757863165 and parameters: {'n_estimators': 605, 'learning_rate': 0.19340488035614609, 'max_depth': 9, 'subsample': 0.668659891625869, 'colsample_bytree': 0.6979198144188306, 'min_child_weight': 8}. Best is trial 16 with value: 0.41048199150367504.


Trial 19 | RMSE: 0.4207 | params: {'n_estimators': 605, 'learning_rate': 0.19340488035614609, 'max_depth': 9, 'subsample': 0.668659891625869, 'colsample_bytree': 0.6979198144188306, 'min_child_weight': 8}
Best RMSE: 0.41048199150367504
Best params:
  n_estimators: 700
  learning_rate: 0.10693756760362194
  max_depth: 3
  subsample: 0.7303402275472366
  colsample_bytree: 0.6006076505440412
  min_child_weight: 9


# 7. Final Model Training
Train the final model with best Optuna parameters on the full training set and evaluate on the holdout test set.

In [7]:
final_params = dict(best_params)
final_params.update({
    "random_state": 42,
    "objective": "reg:squarederror",
    "tree_method": "hist",
    "device": "cuda"
})

final_model = XGBRegressor(**final_params)
final_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=100)

test_pred = final_model.predict(X_test)
rmse = float(np.sqrt(mean_squared_error(y_test, test_pred)))
mae = float(mean_absolute_error(y_test, test_pred))
mape = float(np.mean(np.abs((y_test - test_pred) / y_test.replace(0, np.nan))) * 100)

print(f"RMSE: {rmse:.6f}")
print(f"MAE:  {mae:.6f}")
print(f"MAPE: {mape:.4f}%")

print(f"Retraining on full dataset...")
X_full = pd.concat([X_train, X_test])
y_full = pd.concat([y_train, y_test])
final_model.fit(X_full, y_full, verbose=100)
print("Final model trained on full dataset")



# AUTO SAVE immediately after training
from pathlib import Path
import joblib, json
_save_dir = Path("./")
joblib.dump(final_model, _save_dir / "xgboost_model.pkl")
with open(_save_dir / "xgboost_best_params.json", "w") as f:
    json.dump(best_params, f, indent=2)
with open(_save_dir / "feature_columns.json", "w") as f:
    json.dump(feature_cols, f, indent=2)
print("AUTO SAVED")

[0]	validation_0-rmse:1.55596
[100]	validation_0-rmse:0.50809
[200]	validation_0-rmse:0.50961
[300]	validation_0-rmse:0.51249
[400]	validation_0-rmse:0.51104
[500]	validation_0-rmse:0.51133
[600]	validation_0-rmse:0.51149
[699]	validation_0-rmse:0.51161
RMSE: 0.511613
MAE:  0.317482
MAPE: 18.1994%
Retraining on full dataset...
Final model trained on full dataset
AUTO SAVED


In [8]:
import importlib
import nbformat
importlib.reload(nbformat)
print(nbformat.__version__)

5.10.4


# 8. Feature Importance
Visualize the top 15 most important predictors from the trained XGBoost model.

In [9]:
importance = pd.Series(final_model.feature_importances_, index=feature_cols).sort_values(ascending=False).head(15)

fig_imp = go.Figure(
    go.Bar(
        x=importance.values[::-1],
        y=importance.index[::-1],
        orientation="h",
        marker_color="#2563eb"
    )
)
fig_imp.update_layout(
    title=f"{TICKER} XGBoost Top 15 Feature Importance",
    template="plotly_white",
    xaxis_title="Importance",
    yaxis_title="Feature",
    height=600
)
fig_imp.show()

# 9. Forecast Plot
Plot holdout-period actual vs predicted values and a 30-day future forecast using the latest feature row as a simple forward scenario.

In [ ]:
future_steps = 30
last_row = X_test.iloc[[-1]] if len(X_test) > 0 else X_train.iloc[[-1]]
future_X = pd.concat([last_row] * future_steps, ignore_index=True)
future_pred = final_model.predict(future_X)

# Inverse transform to actual prices
try:
    y_test_actual = preprocessor.inverse_transform(y_test.values)
    test_pred_actual = preprocessor.inverse_transform(test_pred)
    future_pred_actual = preprocessor.inverse_transform(future_pred)
except:
    y_test_actual = y_test.values
    test_pred_actual = test_pred
    future_pred_actual = future_pred

if isinstance(test_df.index, pd.DatetimeIndex):
    future_idx = pd.bdate_range(start=test_df.index[-1] + pd.tseries.offsets.BDay(1), periods=future_steps)
else:
    future_idx = pd.RangeIndex(start=len(test_df), stop=len(test_df) + future_steps)

fig_fc = go.Figure()
fig_fc.add_trace(go.Scatter(x=test_df.index, y=y_test_actual, mode="lines", name="Actual", line=dict(color="#111827", width=2)))
fig_fc.add_trace(go.Scatter(x=test_df.index, y=test_pred_actual, mode="lines", name="Predicted", line=dict(color="#2563eb", width=2)))
fig_fc.add_trace(go.Scatter(x=future_idx, y=future_pred_actual, mode="lines", name="30-Day Forecast", line=dict(color="#10b981", width=2, dash="dash")))
fig_fc.update_layout(
    title=f"{TICKER} Actual vs Predicted + 30-Day Forecast",
    template="plotly_white",
    xaxis_title="Date",
    yaxis_title="Close Price (₹)",
    height=600
)
fig_fc.show()

# 10. Save Artifacts
Persist the trained model, preprocessing artifacts, feature columns, and best hyperparameters under the ticker-specific artifacts directory.

In [11]:
model_path = ARTIFACTS_DIR / "xgboost_model.pkl"
feature_cols_path = ARTIFACTS_DIR / "feature_columns.json"
best_params_path = ARTIFACTS_DIR / "xgboost_best_params.json"

joblib.dump(final_model, model_path)
preprocessor.save_artifacts()

with feature_cols_path.open("w", encoding="utf-8") as f:
    json.dump(feature_cols, f, indent=2)

with best_params_path.open("w", encoding="utf-8") as f:
    json.dump(best_params, f, indent=2)

print("Saved artifacts:")
print(f"- Model: {model_path.resolve()}")
print(f"- Preprocessor scaler: {(ARTIFACTS_DIR / 'scaler.pkl').resolve()}")
print(f"- Feature columns: {feature_cols_path.resolve()}")
print(f"- Best params: {best_params_path.resolve()}")

Saved artifacts:
- Model: C:\Users\gaura\Desktop\FinSight\notebooks\models\xgboost\xgboost_model.pkl
- Preprocessor scaler: C:\Users\gaura\Desktop\FinSight\notebooks\models\xgboost\scaler.pkl
- Feature columns: C:\Users\gaura\Desktop\FinSight\notebooks\models\xgboost\feature_columns.json
- Best params: C:\Users\gaura\Desktop\FinSight\notebooks\models\xgboost\xgboost_best_params.json
